In [1]:
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt

from typing import TypeVar, Generic
from collections import namedtuple

# Define a bounding box structure with coordinates and size
BBox = namedtuple("BBox", ["x_min", "y_min", "width", "height"])

# Define generic type variables for shape and data type
Shape = TypeVar("Shape")
DType = TypeVar("DType")

# Define a typed ndarray subclass to represent a video or image frame
class Frame(np.ndarray, Generic[Shape, DType]):
    """A strongly-typed subclass of np.ndarray representing a frame."""
    pass

In [2]:
class CorrelationFilter:
    def __init__(self, sigma: float = 1.0, num_perturbations: int = 256, min_psr: float = 8.0, seed: int = 6969):
        self._sigma = sigma
        self._num_perturbations = num_perturbations
        self._min_psr = min_psr
        self._bounding_box = None
        np.random.seed(seed)

    def initialize(self, frame: np.ndarray, bbox_xywh) -> bool:
        assert frame.ndim == 2 and frame.dtype == np.uint8, "Invalid input frame"
        self._bounding_box = BBox(*bbox_xywh)
        assert self._bounding_box.width > 0 and self._bounding_box.height > 0, "Invalid bounding box"

        x, y, width, height = self._bbox2roi(bbox_xywh)
        region = frame[y:y + height, x:x + width]
        region = self._preprocess_region(region)

        self._gaussian_fft = np.fft.fft2(self._create_gaussian(width, height))
        region_stack = self._generate_perturbations(region, self._num_perturbations)

        region_fft = np.fft.fft2(region_stack, axes=(1, 2))
        region_fft_conj = np.conj(region_fft)

        gaussian_repeated = np.repeat(self._gaussian_fft[None, :, :], self._num_perturbations, axis=0)
        numerator = gaussian_repeated * region_fft_conj
        denominator = region_fft * region_fft_conj

        self._numerator = np.sum(numerator, axis=0)
        self._denominator = np.sum(denominator, axis=0)
        return True

    def update(self, frame: np.ndarray, learning_rate: float = 0.07, epsilon: float = 1e-5) -> BBox:
        assert frame.ndim == 2 and frame.dtype == np.uint8, "Invalid input frame"
        x, y, width, height = self._bbox2roi(self._bounding_box)
        region = frame[y:y + height, x:x + width]
        cv2.imshow("Region", region)
        region_processed = self._preprocess_region(region)

        region_fft = np.fft.fft2(region_processed)
        filter_fft = self._numerator / (self._denominator + epsilon)
        response = np.fft.ifft2(filter_fft * region_fft).real

        response_norm = cv2.normalize(response, None, 0, 255, cv2.NORM_MINMAX)
        response_uint8 = response_norm.astype(np.uint8)
        response_uint8 = cv2.resize(response_uint8, None, None, 4, 4)
        cv2.imshow("Response Map", response_uint8)

        max_position = np.unravel_index(np.argmax(response), response.shape)
        delta_y = max_position[0] - height // 2
        delta_x = max_position[1] - width // 2

        # Use original BBox dimensions (not ROI's width/height)
        orig_width = self._bounding_box.width
        orig_height = self._bounding_box.height

        x_center = self._bounding_box.x_min + orig_width // 2 + delta_x
        y_center = self._bounding_box.y_min + orig_height // 2 + delta_y

        x_new = x_center - orig_width // 2
        y_new = y_center - orig_height // 2

        new_bbox = self._clamp_bbox(BBox(x_new, y_new, orig_width, orig_height), frame.shape[1], frame.shape[0])

        if new_bbox.width != width or new_bbox.height != height:
            self.initialize(frame, new_bbox)
        else:
            self._update_filter(frame, self._bounding_box, learning_rate)
        
        self._bounding_box = BBox(
            x_min=self._bounding_box.x_min + delta_x,
            y_min=self._bounding_box.y_min + delta_y,
            width=self._bounding_box.width,
            height=self._bounding_box.height
        )


        # PSR computation
        # mean_response = np.mean(response)
        # std_response = np.std(response) + epsilon
        # peak_to_sidelobe_ratio = (response[max_position] - mean_response) / std_response
        return new_bbox

    def _create_gaussian(self, width: int, height: int) -> np.ndarray:
        x_center = width / 2
        y_center = height / 2
        y_indices, x_indices = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
        gaussian = np.exp(-((x_indices - x_center) ** 2 + (y_indices - y_center) ** 2) / (2 * self._sigma))
        gaussian -= gaussian.min()
        gaussian /= (gaussian.max() - gaussian.min() + 1e-8)
        return gaussian

    def _update_filter(self, frame: np.ndarray, bbox: BBox, learning_rate: float):
        x, y, width, height = self._bbox2roi(bbox)
        region = frame[y:y + height, x:x + width]
        region_processed = self._preprocess_region(region)
        region_fft = np.fft.fft2(region_processed)
        region_fft_conj = np.conj(region_fft)

        self._numerator = (1 - learning_rate) * self._numerator + learning_rate * self._gaussian_fft * region_fft_conj
        self._denominator = (1 - learning_rate) * self._denominator + learning_rate * region_fft * region_fft_conj

    def _generate_perturbations(self, region: np.ndarray, num_samples: int, max_angle: float = 18.0) -> np.ndarray:
        height, width = region.shape
        center = (width // 2, height // 2)
        perturbed_regions = []

        for _ in range(num_samples):
            angle = np.random.uniform(-max_angle, max_angle)
            rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
            rotated = cv2.warpAffine(region, rotation_matrix, (width, height),
                                     flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
            perturbed_regions.append(rotated)

        return np.stack(perturbed_regions, axis=0)
    
    def _bbox2roi(self, bbox: BBox) -> BBox:
        bbox = BBox(*bbox)
        # Double the size (expand by 0.5 on all sides)
        x_center = bbox.x_min + bbox.width // 2
        y_center = bbox.y_min + bbox.height // 2

        width_new = bbox.width * 2
        height_new = bbox.height * 2

        x_new = x_center - width_new // 2
        y_new = y_center - height_new // 2

        return BBox(x_new, y_new, width_new, height_new)

    def _preprocess_region(self, region: np.ndarray, epsilon: float = 1e-5) -> np.ndarray:
        region = np.log(region.astype(np.float32) + 1)
        region = (region - region.mean()) / (np.linalg.norm(region) + epsilon)
        return region * self._generate_hann_window(*region.shape)

    @staticmethod
    def _generate_hann_window(height: int, width: int) -> np.ndarray:
        hann_vertical = np.hanning(height)
        hann_horizontal = np.hanning(width)
        return np.outer(hann_vertical, hann_horizontal)

    @staticmethod
    def _clamp_bbox(bbox: BBox, frame_width: int, frame_height: int) -> BBox:
        x, y, width, height = bbox
        x = max(0, min(x, frame_width - 1))
        y = max(0, min(y, frame_height - 1))
        width = min(width, frame_width - x)
        height = min(height, frame_height - y)
        return BBox(x, y, width, height)


In [3]:
def bbox_iou(box1, box2):
    """
    Calculate the Intersection over Union (IoU) of two bounding boxes.
    Boxes should be in (x_min, y_min, width, height) format.
    """
    # Extract coordinates
    b1_x1, b1_y1, b1_w, b1_h = box1
    b2_x1, b2_y1, b2_w, b2_h = box2

    b1_x2 = b1_x1 + b1_w
    b1_y2 = b1_y1 + b1_h
    b2_x2 = b2_x1 + b2_w
    b2_y2 = b2_y1 + b2_h

    # Coordinates of the intersection rectangle
    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)

    # Compute intersection area
    inter_width = max(0, inter_x2 - inter_x1)
    inter_height = max(0, inter_y2 - inter_y1)
    inter_area = inter_width * inter_height

    # Compute union area
    b1_area = b1_w * b1_h
    b2_area = b2_w * b2_h
    union_area = b1_area + b2_area - inter_area

    # Compute IoU (avoid divide-by-zero)
    iou = inter_area / (union_area + 1e-16)

    return iou


In [4]:
path = 'sequences/sunshade/'
ious_per_sequence = {}

groundtruth = pd.read_csv(path+"/groundtruth.txt").to_numpy(int)

frame1 = cv2.imread(path+"color/00000001.jpg", cv2.IMREAD_GRAYSCALE)
filter = CorrelationFilter(sigma=1, num_perturbations=1024)
filter.initialize(frame1, groundtruth[0])
ious = []

for i in range(1, 110):
    frame = cv2.imread(f"{path}color/{i:08d}.jpg", cv2.IMREAD_GRAYSCALE)
    res_box = filter.update(frame)

    frame = cv2.rectangle(frame, (res_box[0], res_box[1]), (res_box[2]+res_box[0], res_box[3]+res_box[1]), (255, 0, 0), 2)
    cv2.imshow("Whole Frame", frame)
    if cv2.waitKey(5) == ord('q'):
        cv2.destroyAllWindows()
        break

    iou = bbox_iou(res_box, groundtruth[i-1])
    ious.append(iou)

ious_per_sequence = np.mean(ious)
print("IOUS : ", np.mean(ious))
cv2.destroyAllWindows()

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/jrosa/AGH_FILES/ZAW-2025S/.venv/lib/python3.12/site-packages/cv2/qt/plugins"


IOUS :  0.6057294008053694
